# Module 12 — Modules, Packages, and Environments

**Part III: Intermediate** · Level: Intermediate · Prerequisites: Module 8

## What you'll learn

- Importing code with `import` and `from … import`
- Aliasing, relative imports, and the module search path
- Structuring code into modules and packages
- The `if __name__ == "__main__"` guard
- Managing dependencies with `pip`, virtual environments, and `requirements.txt`

---

## 12.1 Modules and imports

A **module** is simply a `.py` file. Importing it runs it once and gives you a namespace of its names.

In [ ]:
import math
print(math.sqrt(16))

from math import sqrt, pi          # bring specific names in directly
print(sqrt(25), pi)

import math as m                   # alias
print(m.pi)

from math import factorial as fact
print(fact(5))

| Form | When to use |
|---|---|
| `import module` | default; keeps names qualified (`math.sqrt`) |
| `from module import name` | a few frequently used names |
| `import module as alias` | long names or conventions (`import numpy as np`) |
| `from module import *` | **avoid** — pollutes your namespace |

By convention, imports go at the **top** of the file in three groups: standard library, third-party, then your own modules.

In [ ]:
import random
print(0 <= random.randint(0, 9) <= 9)

## 12.2 Where Python looks for modules

Python searches `sys.path` in order: the script's directory, installed packages, then the standard library.

In [ ]:
import sys

print(len(sys.path) > 0)
print(sys.version.split()[0])

If a module of yours shares a name with a standard one (for example `random.py`), yours can **shadow** it and cause baffling errors. Avoid reusing standard-library names.

## 12.3 A module you write yourself

Let's create a module on the fly, import it, and look at the `__name__` guard.

In [ ]:
import sys
import tempfile
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "mytools.py").write_text('''
"""Small helper module."""

def shout(text):
    return text.upper() + "!"

def main():
    print("running as a script:", shout("hello"))

print("module name is:", __name__)

if __name__ == "__main__":
    main()
''')

sys.path.insert(0, str(workdir))
import mytools                      # prints: module name is: mytools
print(mytools.shout("hi"))          # HI!

## 12.4 `__name__` and the main guard

Every module has a `__name__`. It equals `"__main__"` when the file is **run directly** and the module's own name when it is **imported**. The guard lets a file serve as both a library and a program.

In [ ]:
print(__name__)         # '__main__' in a script or notebook

def main():
    print("main run")

if __name__ == "__main__":
    main()

Importing `mytools` above does *not* call `main()`; running `python mytools.py` does.

## 12.5 Packages

A **package** is a directory of modules. A directory with an `__init__.py` file is a regular package.

```text
project/
├── main.py
└── shapes/
    ├── __init__.py
    ├── circle.py
    └── square.py
```

In [ ]:
# main.py
from shapes.circle import area
import shapes.square as sq

**Relative imports** (inside a package) use dots: `from .circle import area` means "the `circle` module in the same package", and `from ..utils import helper` goes one level up. They work only inside packages, not in a standalone script.

`__init__.py` can be empty, or it can re-export names to give users a tidy public API:

In [ ]:
# shapes/__init__.py
from .circle import area as circle_area
from .square import area as square_area

__all__ = ["circle_area", "square_area"]

## 12.6 Handling optional imports

In [ ]:
try:
    import tomllib               # standard library in Python 3.11+
except ImportError:
    tomllib = None
print("tomllib available:", tomllib is not None)

## 12.7 `pip`, virtual environments, and requirements

**`pip`** installs third-party packages from PyPI. A **virtual environment** gives each project its own isolated set of packages so versions never clash. Run these commands in your terminal (not inside Python):

In [ ]:
%%bash
# create and activate an environment
python -m venv .venv

# Windows (PowerShell / cmd)
.venv\Scripts\activate
# macOS / Linux
source .venv/bin/activate

# install, inspect, and record dependencies
pip install requests
pip list
pip freeze > requirements.txt

# recreate the same environment elsewhere
pip install -r requirements.txt

# leave the environment
deactivate

Good habits:

- One virtual environment per project; never install project packages globally.
- Commit `requirements.txt`; do **not** commit the `.venv` folder.
- Prefer `python -m pip ...` so you know which interpreter you are installing into.

## 12.8 Organising a real project

```text
my_app/
├── README.md
├── requirements.txt
├── src/
│   └── my_app/
│       ├── __init__.py
│       ├── core.py
│       └── cli.py
└── tests/
    └── test_core.py
```

Keep modules focused on one job, keep side effects out of import time, and give every public function a docstring.

## Common pitfalls

- Naming your file `random.py`, `math.py`, or `test.py` and shadowing the real module.
- Circular imports (A imports B, B imports A) — restructure or import inside a function.
- Running `python package/module.py` and then wondering why relative imports fail; use `python -m package.module`.
- Installing packages into the wrong interpreter.

## Practice

1. Create `temperature.py` with `c_to_f()` and `f_to_c()`, and a `main()` demo guarded by `if __name__ == "__main__"`. Import it from another file.
2. Make a virtual environment, install `requests`, and generate a `requirements.txt`.
3. Sketch the package layout for a to-do application with storage, models, and a command-line interface.